In [ ]:
from collections import defaultdict

import json
import numpy as np
import pandas as pd
import polars as pl


In [ ]:
interactions_dataset_path = '../data/Beauty/Beauty_5.json'
metadata_path = '../data/Beauty/metadata.json'

interactions_output_path = '../data/Beauty/inter.json'
texts_output_path = '../data/Beauty/item_texts.parquet'

In [ ]:
import os

BEAUTY_5_URL = (
    'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz'
)

if not os.path.exists(interactions_dataset_path):
    raise FileNotFoundError(
        f'Interactions file not found: {interactions_dataset_path}, download it from {BEAUTY_5_URL}'
    )

df = defaultdict(list)

with open(interactions_dataset_path, 'r') as f:
    for line in f.readlines():
        review = json.loads(line)
        df['user_id'].append(review['reviewerID'])
        df['item_id'].append(review['asin'])
        df['timestamp'].append(review['unixReviewTime'])

print(f'Number of events: {len(df["user_id"])}')

df = pl.from_dict(df)

In [ ]:
df.head()

In [ ]:
filtered_df = df.clone()

In [ ]:
# Processing dataset to get core-5 state in case full dataset is provided
is_changed = True
threshold = 5
good_users = set()
good_items = set()

while is_changed:
    user_counts = filtered_df.group_by('user_id').agg(
        pl.len().alias('user_count'),
    )
    item_counts = filtered_df.group_by('item_id').agg(
        pl.len().alias('item_count'),
    )

    good_users = user_counts.filter(pl.col('user_count') >= threshold).select(
        'user_id',
    )
    good_items = item_counts.filter(pl.col('item_count') >= threshold).select(
        'item_id',
    )

    old_size = len(filtered_df)

    new_df = filtered_df.join(good_users, on='user_id', how='inner')
    new_df = new_df.join(good_items, on='item_id', how='inner')

    new_size = len(new_df)

    filtered_df = new_df
    is_changed = old_size != new_size


In [ ]:
unique_values = filtered_df["user_id"].unique(maintain_order=True).to_list()
user_ids_mapping = {value: i for i, value in enumerate(unique_values)}

filtered_df = filtered_df.with_columns(
    pl.col("user_id").replace_strict(user_ids_mapping)
)

unique_values = filtered_df["item_id"].unique(maintain_order=True).to_list()
item_ids_mapping = {value: i for i, value in enumerate(unique_values)}

filtered_df = filtered_df.with_columns(
    pl.col("item_id").replace_strict(item_ids_mapping)
)

filtered_df.head()

In [ ]:
item_ids_mapping_df = pl.from_dict({
    'old_item_id': list(item_ids_mapping.keys()),
    'new_item_id': list(item_ids_mapping.values())
})
item_ids_mapping_df.head()

In [ ]:
filtered_df.head()

In [ ]:
filtered_df = filtered_df.sort(["user_id", "timestamp"])

grouped_filtered_df = filtered_df.group_by("user_id", maintain_order=True).agg(pl.all())

In [ ]:
item_ids_mapping_df.head()

In [ ]:
grouped_filtered_df.head()

In [ ]:
print('Users count:', filtered_df.select('user_id').unique().shape[0])
print('Items count:', filtered_df.select('item_id').unique().shape[0])
print('Actions count:', filtered_df.shape[0])
print('Avg user history len:', np.mean(list(map(lambda x: x[0], grouped_filtered_df.select(pl.col('item_id').list.len()).rows()))))

In [ ]:
json_data = {}
for user_id, item_ids, _ in grouped_filtered_df.iter_rows():
    json_data[user_id] = item_ids

with open(interactions_output_path, 'w') as f:
    json.dump(json_data, f, indent=2)

## Подготовка исходных текстов товаров

Получение эмбеддингов вынесено в `ContentEmbeddings.ipynb`: один код для исходных и обогащённых описаний.


In [ ]:
import os

METADATA_URL = (
    'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Beauty.json.gz'
)

def getDF(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'Metadata file not found: {path}, download it from {METADATA_URL}'
        )

    i = 0
    df = {}
    with open(path, 'r') as f:
        for line in f.readlines():
            df[i] = eval(line)
            i += 1

    return pd.DataFrame.from_dict(df, orient="index")

df = getDF(metadata_path)
df.head()

In [ ]:
def preprocess(row: pd.Series):
    row = row.fillna("None")
    return f"Title: {row['title']}. Categories: {', '.join(row['categories'][0])}. Description: {row['description']}."


def get_data(metadata_df, item_ids_mapping_df):
    filtered_df = metadata_df.join(
        item_ids_mapping_df, 
        left_on="asin", 
        right_on='old_item_id', 
        how="inner"
    ).select(pl.col('new_item_id'), pl.col('asin'), pl.col('title'), pl.col('description'), pl.col('categories'))

    filtered_df = filtered_df.to_pandas()
    filtered_df["combined_text"] = filtered_df.apply(preprocess, axis=1)

    return filtered_df


In [ ]:
data = get_data(pl.from_pandas(df), item_ids_mapping_df)

In [ ]:
item_texts = data[['new_item_id', 'asin', 'combined_text']].rename(
    columns={'new_item_id': 'item_id', 'combined_text': 'text'},
).sort_values('item_id')
assert len(item_texts) == len(item_ids_mapping)
assert item_texts['item_id'].is_unique and item_texts['asin'].is_unique
item_texts.to_parquet(texts_output_path, index=False)
item_texts.head()
